# Tutorial 1: QuadraSHAP for tree models

This notebook explains one prediction from a small scikit-learn decision tree. We will:

1. compute exact path-dependent TreeSHAP values with QuadraSHAP;
2. implement the same Shapley game by enumerating every feature coalition;
3. verify that the two answers agree and reconstruct the prediction; and
4. compare their execution times.

> Run this notebook from an environment where the repository has been installed with `python -m pip install -e .`.

## What game are we explaining?

QuadraSHAP currently implements **tree-path-dependent** missingness. If a split feature belongs to coalition `S`, the observation follows its usual branch. If the feature is absent, both branches are averaged using the fractions of training weight that reached the child nodes. This is the same game used by path-dependent TreeSHAP.

## The paper's central reduction

The ordinary Shapley formula averages feature `i`'s marginal contribution over every coalition that excludes it:

$$
\phi_i(v)=\sum_{S\subseteq D\setminus\{i\}}
\frac{|S|!(d-|S|-1)!}{d!}
\left[v(S\cup\{i\})-v(S)\right].
$$

That is a sum over $2^{d-1}$ coalitions. QuadraSHAP becomes possible when a game factorizes as $v(S)=\prod_{j\in S}u_j$. Its marginal contribution is then $(u_i-1)\prod_{j\in S}u_j$.

The key step in Proposition 2 of the paper is to rewrite the factorial Shapley weight as a Beta integral:

$$
\frac{s!(d-s-1)!}{d!}=\int_0^1 t^s(1-t)^{d-s-1}\,dt.
$$

After moving the coalition sum inside the integral, each feature $j\ne i$ contributes either $(1-t)$ when it is absent or $tu_j$ when it is present. Expanding all those binary choices is exactly a product, so the exponential sum collapses to

$$
\boxed{\phi_i(v)=(u_i-1)\int_0^1\prod_{j\ne i}[(1-t)+tu_j]\,dt.}
$$

The integrand is a polynomial of degree at most $d-1$. An $m_q$-node Gauss-Legendre rule integrates every polynomial through degree $2m_q-1$, which proves exactness when $m_q\ge\lceil d/2\rceil$ (Proposition 3). Smaller budgets are approximations whose error is shown in the paper to decay geometrically under the stated analytic bound.

In [1]:
import math
import time

import numpy as np
from sklearn.datasets import make_regression
from sklearn.tree import DecisionTreeRegressor

from quadrashap import TreeExplainer
from quadrashap._cpp_ext import HAS_CPP_EXT

np.set_printoptions(precision=6, suppress=True)

## Train a small decision tree

Twelve features give the naive baseline `2**12 = 4,096` coalitions: enough to expose the scaling difference while remaining tutorial-friendly.

In [2]:
N_FEATURES = 12

X, y = make_regression(
    n_samples=800,
    n_features=N_FEATURES,
    n_informative=8,
    noise=0.1,
    random_state=0,
)
model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=5,
    random_state=0,
).fit(X, y)

x = X[0]
n_leaves = model.get_n_leaves()
print(f"features={N_FEATURES}, leaves={n_leaves}, coalitions={2**N_FEATURES:,}")

features=12, leaves=54, coalitions=4,096


## How a tree becomes a weighted sum of product games

For a leaf $\ell$, let $E_\ell$ be its root-to-leaf edges, $\nu_\ell$ its prediction, and $p_e$ the fraction of training weight taking edge $e$. For the observation being explained, define

$$
c_e(x)=\begin{cases}1/p_e,&x\text{ follows edge }e,\\0,&\text{otherwise,}\end{cases}
\qquad
q_{j,\ell}(x)=\prod_{e\in E_\ell:\,\kappa(e)=j}c_e(x).
$$

Here $\kappa(e)$ is the feature used at the split above edge $e$. If feature $j$ is known, $q_{j,\ell}$ cancels the branch probabilities for its matching path edges when the observation satisfies them, and makes the leaf contribution zero otherwise. If $j$ is unknown, those branch probabilities remain. Therefore the path-dependent coalition game is

$$
v_x(S)=\sum_{\ell\in L}
\underbrace{\nu_\ell\prod_{e\in E_\ell}p_e}_{\text{empty-coalition leaf weight}}
\prod_{j\in S}q_{j,\ell}(x).
$$

Every leaf is thus one product game, and Shapley linearity lets us add their attributions. Importantly, a leaf only depends on the **distinct** split features on its path. The paper calls the maximum such count $\eta$, the effective path dimension. Usually $\eta\ll d$, and repeated splits on one feature count only once. Exact tree quadrature therefore needs $m_q=\lceil\eta/2\rceil$, not $\lceil d/2\rceil$.

In [3]:
def max_distinct_path_features(model):
    """Return eta: the largest distinct-feature count on any leaf path."""
    tree = model.tree_
    active_features = set()

    def visit(node):
        left = int(tree.children_left[node])
        right = int(tree.children_right[node])
        if left == -1 and right == -1:
            return len(active_features)

        feature = int(tree.feature[node])
        added_here = feature not in active_features
        if added_here:
            active_features.add(feature)
        result = max(visit(left), visit(right))
        if added_here:
            active_features.remove(feature)
        return result

    return visit(0)


ETA = max_distinct_path_features(model)
M_Q_EXACT = max(1, (ETA + 1) // 2)
print(f"ambient features d:           {N_FEATURES}")
print(f"tree depth h:                 {model.get_depth()}")
print(f"effective path dimension eta: {ETA}")
print(f"exact quadrature nodes:       {M_Q_EXACT}")

ambient features d:           12
tree depth h:                 6
effective path dimension eta: 5
exact quadrature nodes:       3


## Explain with QuadraSHAP

The direct `quadrature_tree` solver uses the exact default quadrature order. `expected_value` is the path-dependent value of the empty coalition.

In [4]:
explainer = TreeExplainer(
    model,
    tree_solver="quadrature_tree",
    use_cpp=HAS_CPP_EXT,
)

phi_quadra = explainer.shap_values(x[None, :], check_additivity=True)[0]
baseline_quadra = float(np.asarray(explainer.expected_value).reshape(-1)[0])
prediction = float(model.predict(x[None, :])[0])
reconstruction = baseline_quadra + phi_quadra.sum()

backend = "C++" if HAS_CPP_EXT else "pure Python"
print(f"backend:        {backend}")
print(f"baseline:       {baseline_quadra: .8f}")
print(f"sum(phi):       {phi_quadra.sum(): .8f}")
print(f"prediction:     {prediction: .8f}")
print(f"reconstruction: {reconstruction: .8f}")

backend:        pure Python
baseline:       -17.43020980
sum(phi):        68.77696811
prediction:      51.34675831
reconstruction:  51.34675831


## How the direct tree solver avoids repeated leaf work

Applying the product-game formula independently to every leaf would cost $O(m_q|L|\eta)$: every leaf, every quadrature node, and every path feature. The optimized algorithm in Section 4.2 and Appendix C of the paper reorganizes the same sum into one depth-first traversal.

At quadrature node $\tau_r$, define

$$a_r(q)=1-\tau_r+\tau_r q,\qquad s_r(q)=\frac{q-1}{a_r(q)}.$$

The traversal maintains the current factor `q[j]` for each feature and a running path product $B_r$ for every quadrature node. Crossing an edge changes only the feature used by that split, so all $B_r$ values update in $O(m_q)$ work. A leaf returns its prediction multiplied by those running products. On the way back up, child values are aggregated into subtree totals.

If a feature appears several times along a path, the changes in $s_r(q)$ telescope: intermediate states cancel, leaving precisely that feature's leaf contribution. This converts leaf-feature calculations into edge-local updates. Every edge is visited once down and once up, giving

$$O(m_q|L|)\text{ work},\qquad O(\eta|L|)\text{ in the exact regime},$$

instead of coalition enumeration or an $|L|\times\eta$ table. The current implementation exposes this algorithm as `tree_solver="quadrature_tree"`.

A second benefit is numerical: QuadraSHAP evaluates the polynomial at stable Gauss-Legendre nodes and does not recover coefficients by inverting an increasingly ill-conditioned Vandermonde system. The paper also describes log-space product updates for extreme path products, where multiplication and division become addition and subtraction of log-magnitudes with signs tracked separately.

## Exact and approximate quadrature budgets

The default uses the proven exact threshold. Supplying a smaller `m_q` trades accuracy for speed. On this small tree we can compare every smaller budget with the exact attribution vector.

In [5]:
print(f"{'m_q':>4} {'L2 distance to exact':>24} {'efficiency residual':>24}")
for m_q in range(1, M_Q_EXACT + 1):
    budget_explainer = TreeExplainer(
        model,
        tree_solver="quadrature_tree",
        use_cpp=HAS_CPP_EXT,
        m_q=m_q,
    )
    phi_mq = budget_explainer.shap_values(
        x[None, :], check_additivity=False
    )[0]
    distance = np.linalg.norm(phi_mq - phi_quadra)
    residual = abs(baseline_quadra + phi_mq.sum() - prediction)
    print(f"{m_q:4d} {distance:24.3e} {residual:24.3e}")

 m_q     L2 distance to exact      efficiency residual
   1                1.181e+01                2.210e+01
   2                6.748e-02                5.701e-02
   3                0.000e+00                2.842e-14


## A naive exact baseline

The code below makes the definition explicit. It evaluates `v(S)` for every coalition and then applies the factorial Shapley weights. Its runtime grows exponentially with the number of features.

In [6]:
def path_dependent_tree_value(model, x, known_mask):
    """Evaluate v(S) for one regression tree and an integer coalition mask."""
    tree = model.tree_
    node_weight = np.asarray(tree.weighted_n_node_samples, dtype=float)

    def visit(node):
        left = int(tree.children_left[node])
        right = int(tree.children_right[node])
        if left == -1 and right == -1:
            return float(np.asarray(tree.value[node]).reshape(-1)[0])

        feature = int(tree.feature[node])
        if known_mask & (1 << feature):
            child = left if x[feature] <= tree.threshold[node] else right
            return visit(child)

        parent_weight = node_weight[node]
        p_left = node_weight[left] / parent_weight
        p_right = node_weight[right] / parent_weight
        return p_left * visit(left) + p_right * visit(right)

    return visit(0)


def naive_tree_shapley(model, x):
    """Exact path-dependent TreeSHAP by exhaustive coalition enumeration."""
    d = int(model.n_features_in_)
    coalition_values = np.empty(1 << d, dtype=float)
    for mask in range(1 << d):
        coalition_values[mask] = path_dependent_tree_value(model, x, mask)

    factorial = [math.factorial(k) for k in range(d + 1)]
    phi = np.zeros(d, dtype=float)
    for feature in range(d):
        feature_bit = 1 << feature
        for mask in range(1 << d):
            if mask & feature_bit:
                continue
            size = mask.bit_count()
            weight = factorial[size] * factorial[d - size - 1] / factorial[d]
            phi[feature] += weight * (
                coalition_values[mask | feature_bit] - coalition_values[mask]
            )

    return phi, coalition_values[0]

In [7]:
phi_naive, baseline_naive = naive_tree_shapley(model, x)

np.testing.assert_allclose(phi_quadra, phi_naive, rtol=1e-8, atol=1e-8)
np.testing.assert_allclose(baseline_quadra, baseline_naive, rtol=1e-10, atol=1e-10)
np.testing.assert_allclose(reconstruction, prediction, rtol=1e-10, atol=1e-10)

print(f"maximum attribution difference: {np.max(np.abs(phi_quadra - phi_naive)):.3e}")
print(f"additivity error:               {abs(reconstruction - prediction):.3e}")

print("\nFive largest attributions:")
for feature in np.argsort(np.abs(phi_quadra))[::-1][:5]:
    print(f"  feature {feature:2d}: {phi_quadra[feature]: .6f}")

maximum attribution difference: 2.700e-13
additivity error:               2.842e-14

Five largest attributions:
  feature  9:  71.154586
  feature  3:  71.033209
  feature  6: -69.597560
  feature  2: -34.353681
  feature 10:  24.587364


## Timing comparison

The explainer is constructed before timing, so this compares repeated explanations of one observation. QuadraSHAP is warmed up once; the naive implementation has no compiled or JIT warm-up.

In [8]:
def timed(callable_, repeats):
    durations = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = callable_()
        durations.append(time.perf_counter() - start)
    return result, float(np.median(durations))


_ = explainer.shap_values(x[None, :], check_additivity=False)
_, quadra_seconds = timed(
    lambda: explainer.shap_values(x[None, :], check_additivity=False),
    repeats=20,
)
_, naive_seconds = timed(lambda: naive_tree_shapley(model, x), repeats=3)

print(f"QuadraSHAP ({backend:11s}): {quadra_seconds * 1e3:9.3f} ms")
print(f"Naive enumeration:         {naive_seconds * 1e3:9.3f} ms")
print(f"Measured speedup:          {naive_seconds / quadra_seconds:9.1f}x")

QuadraSHAP (pure Python):     0.873 ms
Naive enumeration:           105.306 ms
Measured speedup:              120.6x


## Beyond this toy comparison

The naive baseline is useful for understanding correctness but is not a competitive tree explainer. Section 5.2 of *QuadraSHAP: Stable and Scalable Shapley Values for Product Games via Gauss-Legendre Quadrature* compares against TreeSHAP, FastTreeSHAP, Linear TreeSHAP, and shapiq.

On single-threaded synthetic random forests with 10 or 100 features and up to 100,000 leaves, the paper reports QuadraSHAP as the fastest numerically stable method in every tested configuration; at 100,000 leaves it is about 3-5x faster than `shap` while retaining efficiency residuals near floating-point precision. On five 5,000-dimensional text datasets, with forests ranging from roughly 24,000 to 181,000 leaves and depths up to 100, it reports 15-30x speedups over `shap` and FastTreeSHAP and stable efficiency on all five datasets. See Tables 1-2 and the full values in Appendix F. Those paper results use a controlled benchmark environment; the timing above measures only this notebook's small model on your machine.

## Takeaway

Both methods calculate the same exact path-dependent Shapley values. The naive method exists here to expose the definition, but its `2**d` coalitions quickly become impractical. QuadraSHAP instead evaluates a small Gauss–Legendre quadrature while traversing the tree. Your timing will depend on the machine and on whether the optional C++ extension is available.